# CNN_5_7ch_Exp17A  Bias Adjustment ของคลาส N1 ใน Soft Voting Ensemble

# 1.Import

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_fscore_support,
)

# 2. Setting

In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/SleepStageClassificationProject")

EXP13_DIRS = {
    42: PROJECT_DIR / "results" / "CNN_5_7ch_Exp13_seed42",
    43: PROJECT_DIR / "results" / "CNN_5_7ch_Exp13_Seed43",
    44: PROJECT_DIR / "results" / "CNN_5_7ch_Exp13_Seed44",
}
RESULT_DIR = PROJECT_DIR / "results" / "CNN_5_7ch_Exp17_bias_SV"
assert RESULT_DIR not in EXP13_DIRS.values()
RESULT_DIR.mkdir(parents=True, exist_ok=True)

STAGES = ["W", "N1", "N2", "N3", "R"]
LABELS = [0, 1, 2, 3, 4]
PROB_COLS = ["prob_" + s for s in STAGES]
KEY_COLS = ["subject", "epoch_start", "epoch_end", "actual"]
SEEDS = [42, 43, 44]
SELECTED_BIAS = None

# 3.โหลด CSV และตรวจความตรงกัน

In [ ]:
saved = {
    seed: pd.read_csv(EXP13_DIRS[seed] / "val_predictions.csv")
    for seed in SEEDS
}
base = saved[42]
assert len(base) == 2878

for seed, df in saved.items():
    assert len(df) == len(base)
    for col in KEY_COLS:
        assert (df[col].astype(str).to_numpy() == base[col].astype(str).to_numpy()).all(), f"Seed {seed}: {col} ไม่ตรงกัน"

    prob = df[PROB_COLS].to_numpy()
    pred = np.array(STAGES)[prob.argmax(axis=1)]

    assert np.isfinite(prob).all()
    assert np.allclose(prob.sum(axis=1), 1, atol=1e-3)
    assert (pred == df["predicted"].to_numpy()).all()
    assert np.allclose(df["confidence"], prob.max(axis=1), atol=1e-5)
    assert (df["correct"].astype(str).str.upper().to_numpy() == np.where(df["actual"] == df["predicted"], "TRUE", "FALSE")).all()

y_val = base["actual"].map(dict(zip(STAGES, LABELS))).to_numpy()
assert not pd.isna(y_val).any()
y_val = y_val.astype(int)
probs = {seed: df[PROB_COLS].to_numpy() for seed, df in saved.items()}

print("ตัวอย่างตรงกัน:", len(y_val), "sequences,", base["subject"].nunique(), "subjects")

# 4.Soft Voting

In [ ]:
mean_probs = np.mean(np.stack([probs[s] for s in SEEDS], axis=0), axis=0)
soft_pred = mean_probs.argmax(axis=1)

# 5.ตรวจผล Soft Voting เดิม

In [ ]:
def evaluate(y_true, y_pred):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=LABELS,
        zero_division=0
    )

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro-Precision": precision.mean(),
        "Macro-Recall": recall.mean(),
        "Macro-F1": f1.mean(),
        "Kappa": cohen_kappa_score(y_true, y_pred),
        **{f"{s}_Precision": precision[i] for i, s in enumerate(STAGES)},
        **{f"{s}_Recall": recall[i] for i, s in enumerate(STAGES)},
        **{f"{s}_F1": f1[i] for i, s in enumerate(STAGES)},
        **{f"{s}_Support": support[i] for i, s in enumerate(STAGES)},
    }

baseline = evaluate(y_val, soft_pred)

print("Exp13 Soft Voting — Validation")
display(pd.DataFrame([baseline]).round(4))

In [ ]:
pd.DataFrame([baseline]).to_csv(RESULT_DIR / "baseline_soft_voting.csv", index=False)

cm = confusion_matrix(y_val, soft_pred, labels=LABELS)

pd.DataFrame(cm, index=[f"Actual_{s}" for s in STAGES],
             columns=[f"Predicted_{s}" for s in STAGES]).to_csv(
    RESULT_DIR / "baseline_confusion_matrix.csv"
)

# 6. ทดลอง Bias ของ N1 บน Soft Voting

In [ ]:
def soft_vote_with_bias(bias):
    scores = np.log(np.clip(mean_probs, 1e-12, 1.0))
    scores[:, 1] += bias
    return scores.argmax(axis=1)

pred_zero = soft_vote_with_bias(0.0)

assert np.array_equal(pred_zero, soft_pred), \
    "Bias = 0 ไม่ตรงกับ Soft Voting เดิม กรุณาตรวจสอบก่อน"

print("Bias = 0 ตรงกับ Soft Voting เดิม")

# 7. Bias Grid Search บน Validation เท่านั้น

In [ ]:
bias_values = np.round(np.arange(-1.0, 1.001, 0.25), 2)

results = []

for bias in bias_values:
    pred = soft_vote_with_bias(bias)
    row = {"Bias": bias}
    row.update(evaluate(y_val, pred))
    results.append(row)

results_df = pd.DataFrame(results)

display(
    results_df[
        ["Bias", "Accuracy", "Macro-Precision", "Macro-Recall",
         "Macro-F1", "Kappa",
         "N1_Precision", "N1_Recall", "N1_F1",
         "W_F1", "N2_F1", "N3_F1", "R_F1"]
    ].round(4)
)

results_df.to_csv(RESULT_DIR / "validation_bias_results.csv",index=False)

# 7.1 คัดเลือก Bias จาก Validation ตามเกณฑ์ที่กำหนด

In [ ]:
eligible = results_df[
    (results_df["Macro-F1"] >= baseline["Macro-F1"] + 0.005) &
    (results_df["Accuracy"] >= baseline["Accuracy"] - 0.005) &
    (results_df["W_F1"] >= baseline["W_F1"] - 0.02) &
    (results_df["N2_F1"] >= baseline["N2_F1"] - 0.02) &
    (results_df["N3_F1"] >= baseline["N3_F1"] - 0.02) &
    (results_df["R_F1"] >= baseline["R_F1"] - 0.02)
].copy()

if eligible.empty:
    SELECTED_BIAS = 0.0
    print("ไม่มี Bias ที่ผ่านเกณฑ์ คงใช้ Baseline: Bias = 0.0")
else:
    eligible["Abs_Bias"] = eligible["Bias"].abs()
    eligible = eligible.sort_values(
        ["Macro-F1", "Abs_Bias"],
        ascending=[False, True]
    )
    SELECTED_BIAS = float(eligible.iloc[0]["Bias"])
    print("Bias ที่เลือกจาก Validation:", SELECTED_BIAS)

# 8. กราฟผลตามค่า Bias

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(results_df["Bias"], results_df["Macro-F1"],
         marker="o", label="Macro-F1")
plt.plot(results_df["Bias"], results_df["N1_F1"],
         marker="o", label="N1 F1")
plt.plot(results_df["Bias"], results_df["Accuracy"],
         marker="o", label="Accuracy")

plt.axvline(0, linestyle="--", alpha=0.6)
plt.xlabel("N1 Bias")
plt.ylabel("Score")
plt.title("Exp17A (Soft) N1 Bias Adjustment on Validation")
plt.xticks(bias_values)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(RESULT_DIR / "validation_bias_plot.png",dpi=300,bbox_inches="tight")
plt.show()



# 9. บันทึกผล Bias ที่เลือกจาก Validation

In [ ]:
selected_pred = soft_vote_with_bias(SELECTED_BIAS)
selected_metrics = evaluate(y_val, selected_pred)

print("Selected Bias:", SELECTED_BIAS)
display(pd.DataFrame([selected_metrics]).round(4))

pd.DataFrame([{
    "Selected_Bias": SELECTED_BIAS,
    **selected_metrics
}]).to_csv(
    RESULT_DIR / "selected_bias_validation_metrics.csv",
    index=False
)

pd.DataFrame({
    "subject": base["subject"],
    "epoch_start": base["epoch_start"],
    "epoch_end": base["epoch_end"],
    "actual": base["actual"],
    "predicted": np.array(STAGES)[selected_pred],
    "correct": selected_pred == y_val
}).to_csv(
    RESULT_DIR / "selected_bias_validation_predictions.csv",
    index=False
)

cm = confusion_matrix(y_val, selected_pred, labels=LABELS)

pd.DataFrame(cm,index=[f"Actual_{s}" for s in STAGES],columns=[f"Predicted_{s}" for s in STAGES]).to_csv(RESULT_DIR / "selected_bias_confusion_matrix.csv")